# debug nik_mri style fit on dce data
poking at the center-blob artifact. runs a short training and logs checks along the way.

In [2]:
%matplotlib inline
import os, sys
# make sure we import from the repo
REPO = '/scratch/rnga/vvpshenov/DCE_NIK'
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)

import numpy as np
import torch
import matplotlib
# force inline backend in case jupyter picked up Agg from somewhere
try:
    matplotlib.use('module://matplotlib_inline.backend_inline')
except Exception:
    pass
import matplotlib.pyplot as plt

from nik_io import load_event
from nik_loss import HDRLossFF
from nik_model import NIK_MRI_SIREN_REIM
from nik_mri_style import (
    DynamicSliceDataset,
    build_nik_mri_dce_dataset,
    coil_combine,
    ifft2c,
    predict_cartesian_kspace,
    prepare_coil_sensitivity_maps,
)
from nik_recon import ifft1d_kz_to_z
from nik_train import prepare_tensors

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('matplotlib backend:', matplotlib.get_backend())

device: cuda
matplotlib backend: Agg


## 1. load data

In [3]:
DATA_FILE = '/scratch/rnga/vvpshenov/XCAT-ERIC/results/simulation_results_20260326T114914_rad.mat'

event = load_event(DATA_FILE, load_images=True, load_coil_maps=True)
# load_event returns (T, C, S, RO) despite the old docstring. match the train script convention.
k_np = np.transpose(event['k'], (0, 2, 1, 3))
traj_np = np.transpose(event['traj'], (0, 2, 1, 3))
T, S, C, RO = k_np.shape
print(f'k shape  (T,S,C,RO) = ({T},{S},{C},{RO})')
print('traj shape        =', traj_np.shape)
print('coil_maps shape   =', None if event.get('coil_maps') is None else event['coil_maps'].shape)

k_t, traj_t, scales, dims, _ = prepare_tensors(k_np, traj_np, data_device='cpu')
sx, sy, sz = (float(s) for s in scales)
print(f'scales (sx,sy,sz) = ({sx:.3f}, {sy:.3f}, {sz:.3f})')

k shape  (T,S,C,RO) = (5,2343,8,312)
traj shape        = (5, 2343, 3, 312)
coil_maps shape   = (8, 11, 312, 312)
scales (sx,sy,sz) = (0.500, 0.500, 11.000)


## 2. trajectory sanity — stack of stars assumption
the kz->z ifft assumes that n_slices consecutive spokes share (kx,ky) and differ only in kz. if that's not true the hybrid-space targets are paired with the wrong angles.

In [4]:
kz_vals = traj_t[0, :, 2, 0]
unique_kz = torch.unique(kz_vals)
n_slices = len(unique_kz)
n_ro_per_slice = S // n_slices
print(f'n_slices = {n_slices}, n_ro_per_slice = {n_ro_per_slice}, S/n_slices has remainder = {S % n_slices}')

# peek at first ~2*n_slices spokes. within one 'group' (n_slices spokes)
# kx,ky should stay ~constant and kz should sweep.
print('\nfirst 2*n_slices spokes, at center readout (RO//2):')
print(f"{'spoke':>5} {'kx':>10} {'ky':>10} {'kz':>10}")
for s in range(min(2*n_slices, S)):
    kx_c = traj_t[0, s, 0, RO//2].item()
    ky_c = traj_t[0, s, 1, RO//2].item()
    kz_c = traj_t[0, s, 2, 0].item()
    print(f'{s:>5d} {kx_c:10.4f} {ky_c:10.4f} {kz_c:10.4f}')

n_slices = 11, n_ro_per_slice = 213, S/n_slices has remainder = 0

first 2*n_slices spokes, at center readout (RO//2):
spoke         kx         ky         kz
    0     0.0000     0.0016     6.0000
    1     0.0000     0.0016     7.0000
    2     0.0000     0.0016     5.0000
    3     0.0000     0.0016     8.0000
    4     0.0000     0.0016     4.0000
    5     0.0000     0.0016     9.0000
    6     0.0000     0.0016     3.0000
    7     0.0000     0.0016    10.0000
    8     0.0000     0.0016     2.0000
    9     0.0000     0.0016    11.0000
   10     0.0000     0.0016     1.0000
   11     0.0015    -0.0006     6.0000
   12     0.0015    -0.0006     7.0000
   13     0.0015    -0.0006     5.0000
   14     0.0015    -0.0006     8.0000
   15     0.0015    -0.0006     4.0000
   16     0.0015    -0.0006     9.0000
   17     0.0015    -0.0006     3.0000
   18     0.0015    -0.0006    10.0000
   19     0.0015    -0.0006     2.0000
   20     0.0015    -0.0006    11.0000
   21     0.0015    -0.

In [5]:
# quantify: how much does kx,ky vary within one 'group' of n_slices consecutive spokes?
# if the stack-of-stars assumption holds these should be ~0 and kz std should be large.
n_groups = S // n_slices
groups = torch.arange(S)[:n_groups * n_slices].reshape(n_groups, n_slices)

kx_c = traj_t[0, :, 0, RO//2]
ky_c = traj_t[0, :, 1, RO//2]
kz_c = traj_t[0, :, 2, 0]

within_kx_std = kx_c[groups].std(dim=1).mean().item()
within_ky_std = ky_c[groups].std(dim=1).mean().item()
within_kz_std = kz_c[groups].std(dim=1).mean().item()
print(f'avg within-group std:  kx {within_kx_std:.4g}   ky {within_ky_std:.4g}   kz {within_kz_std:.4g}')
print('-> if kx,ky stds are ~0 and kz std is large, the pipeline assumption is CORRECT')
print('-> otherwise the kz ifft is mixing different angles and you have garbage targets')

avg within-group std:  kx 0   ky 0   kz 3.317
-> if kx,ky stds are ~0 and kz std is large, the pipeline assumption is CORRECT
-> otherwise the kz ifft is mixing different angles and you have garbage targets


## 3. trajectory extent and dc location
check kx,ky span [-1,1] after /(sx,sy) and that the center readout is actually at (0,0).

In [6]:
kx_n = traj_t[..., 0, :] / sx
ky_n = traj_t[..., 1, :] / sy
print(f'kx normalized range: [{kx_n.min():.3f}, {kx_n.max():.3f}]')
print(f'ky normalized range: [{ky_n.min():.3f}, {ky_n.max():.3f}]')
print(f'kx/ky anisotropy sx/sy = {sx/sy:.3f}')

# where is the trajecory center (dc)? look at one spoke, see where |kx|+|ky| is smallest.
spk = 0
r = (kx_n[0, spk]**2 + ky_n[0, spk]**2).sqrt()
r_min_idx = int(r.argmin())
print(f'\nspoke {spk}: min-|k| at readout index {r_min_idx} (RO={RO}, expected ~RO//2 = {RO//2})')
print(f'kx,ky at min-|k|: ({kx_n[0,spk,r_min_idx]:.4g}, {ky_n[0,spk,r_min_idx]:.4g})')

# quick scatter plot of a few spokes
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
for s in range(0, min(400, S), 8):
    ax.plot(kx_n[0, s].cpu().numpy(), ky_n[0, s].cpu().numpy(), '-', lw=0.3, alpha=0.4)
ax.set_aspect('equal'); ax.set_xlabel('kx/sx'); ax.set_ylabel('ky/sy')
ax.set_title('normalised trajectory, subset of spokes')
ax.axhline(0, color='k', lw=0.3); ax.axvline(0, color='k', lw=0.3)
plt.show()

kx normalized range: [-1.000, 1.000]
ky normalized range: [-1.000, 1.000]
kx/ky anisotropy sx/sy = 1.000

spoke 0: min-|k| at readout index 155 (RO=312, expected ~RO//2 = 156)
kx,ky at min-|k|: (-0, -0.003215)


/home/rnga/vvpshenov/tmp/ipykernel_983397/3964562411.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. kz->z ifft + build dataset

In [7]:
k_img_space, n_slices, n_ro_per_slice, _ = ifft1d_kz_to_z(k_t, traj_t, t_frame=0)
print('k_img_space shape (T, ro_per_slice, C, Z, RO) =', tuple(k_img_space.shape))

z_slice_idx = n_slices // 2  # middle slice like the config default (-1)
coords, targets, data_meta = build_nik_mri_dce_dataset(
    k_img_space, traj_t, scales,
    z_slice_idx=z_slice_idx, n_slices=n_slices, target_device='cpu',
)
print('coords:', coords.shape, coords.dtype)
print('targets:', targets.shape, targets.dtype)
print('per-slice point count:', data_meta['n_total'])
print('dataset scale (max |k|):', data_meta['scale'])

k_img_space shape (T, ro_per_slice, C, Z, RO) = (5, 213, 8, 11, 312)
coords: torch.Size([2658240, 4]) torch.float32
targets: torch.Size([2658240, 1]) torch.complex64
per-slice point count: 2658240
dataset scale (max |k|): 1543.7667236328125


In [8]:
# coord / target distribution check
print('coord dim sanity:')
for i, name in enumerate(['t', 'coil', 'kx', 'ky']):
    c = coords[:, i]
    print(f'  {name}: min={c.min().item():+.3f}  max={c.max().item():+.3f}  mean={c.mean().item():+.3f}')

tgt_abs = targets.abs().cpu().numpy().reshape(-1)
qs = np.quantile(tgt_abs, [0.5, 0.9, 0.99, 1.0])
print(f'\n|target| quantiles 0.5/0.9/0.99/max = {qs[0]:.3g} {qs[1]:.3g} {qs[2]:.3g} {qs[3]:.3g}')
print('dynamic range (max / median) =', qs[3] / (qs[0] + 1e-12))

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(tgt_abs, bins=200, log=True); ax[0].set_title('|target| histogram (log)')
r = (coords[:, 2]**2 + coords[:, 3]**2).sqrt().cpu().numpy()
ax[1].scatter(r[::200], tgt_abs[::200], s=0.5, alpha=0.3)
ax[1].set_xlabel('|k|'); ax[1].set_ylabel('|target|'); ax[1].set_yscale('log')
ax[1].set_title('|target| vs radial distance')
plt.tight_layout(); plt.show()

coord dim sanity:
  t: min=-0.800  max=+0.800  mean=+0.000
  coil: min=-1.000  max=+1.000  mean=-0.000
  kx: min=-1.000  max=+1.000  mean=-0.000
  ky: min=-1.000  max=+1.000  mean=+0.000

|target| quantiles 0.5/0.9/0.99/max = 0.000713 0.00908 0.166 1
dynamic range (max / median) = 1402.4622791885677


/home/rnga/vvpshenov/tmp/ipykernel_983397/1949850143.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. coil maps check
wrong axis ordering of csm would make the combined image collapse to a blob.

In [9]:
coil_maps = prepare_coil_sensitivity_maps(event.get('coil_maps'), z_slice_idx, n_coils=C)
print('coil_maps prepared shape:', coil_maps.shape if coil_maps is not None else None)

# rss should be ~1 after normalization (small numerical drift ok)
rss = np.sqrt((np.abs(coil_maps)**2).sum(axis=0))
print(f'rss: min {rss.min():.4g}  max {rss.max():.4g}  mean {rss.mean():.4g}')

# show first few coil maps
ncoils_show = min(6, coil_maps.shape[0])
fig, axes = plt.subplots(2, ncoils_show, figsize=(2.1*ncoils_show, 4.5))
for c in range(ncoils_show):
    axes[0, c].imshow(np.abs(coil_maps[c]), cmap='gray'); axes[0, c].set_title(f'|csm| coil {c}')
    axes[1, c].imshow(np.angle(coil_maps[c]), cmap='twilight'); axes[1, c].set_title(f'∠csm coil {c}')
    for ax in axes[:, c]: ax.axis('off')
plt.tight_layout(); plt.show()

coil_maps prepared shape: (8, 312, 312)
rss: min 1  max 1  mean 1


/home/rnga/vvpshenov/tmp/ipykernel_983397/231349180.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. reference recon from raw data (no model)
if this already shows a center blob, the problem is in the data pipeline, not the network.

In [10]:
from nik_recon import nufft2d_recon

img_size = coil_maps.shape[-2:] if coil_maps is not None else (312, 312)
print('img_size:', img_size)

# get a per-coil complex image via nufft adjoint then combine with csm
coil_imgs_ref = []
for c in range(C):
    img_c = nufft2d_recon(
        k_img_space, traj_t, t_frame=0, coil_idx=c,
        z_slice_idx=z_slice_idx, scales=scales,
        img_size=img_size, n_slices=n_slices, return_complex=True,
    )
    coil_imgs_ref.append(img_c)
coil_imgs_ref = np.stack(coil_imgs_ref, axis=0)  # (C, H, W)

sense_ref = (coil_imgs_ref * np.conj(coil_maps)).sum(axis=0)
rss_ref = np.sqrt((np.abs(coil_imgs_ref)**2).sum(axis=0))

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(np.abs(sense_ref), cmap='gray'); ax[0].set_title('ref |sense combined|'); ax[0].axis('off')
ax[1].imshow(rss_ref, cmap='gray'); ax[1].set_title('ref rss'); ax[1].axis('off')
plt.tight_layout(); plt.show()
print('-> if this looks already like a blob, its not the NN. its the data/pipeline.')

img_size: (312, 312)
-> if this looks already like a blob, its not the NN. its the data/pipeline.


/home/rnga/vvpshenov/tmp/ipykernel_983397/1764017064.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 7. build model and short training
using num_steps=30 outer epochs for debugging (not the 50000 in the full config).

In [11]:
from torch.utils.data import DataLoader

# move dataset to gpu-friendly loader
ds = DynamicSliceDataset(coords, targets)
loader = DataLoader(ds, batch_size=30000, shuffle=True, num_workers=0, pin_memory=(device.type=='cuda'))

def build_model(seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    m = NIK_MRI_SIREN_REIM(coord_dim=4, feature_dim=512, num_layers=8, out_dim=1, omega_0=30.0, ff_scale=1.0)
    return m.to(device)

def run_training(hdr_ff_factor, num_epochs=30, lr=3e-5, log_every=5, seed=0):
    model = build_model(seed=seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = HDRLossFF(sigma=1.0, eps=1e-2, factor=hdr_ff_factor)
    history = []
    for ep in range(1, num_epochs + 1):
        model.train()
        loss_sum = reg_sum = n = 0
        for batch in loader:
            x = batch['coords'].to(device, non_blocking=True)
            y = batch['targets'].to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            pred = model(x)
            loss, reg = crit(pred, y, x)
            loss.backward(); opt.step()
            loss_sum += float(loss.item()); reg_sum += float(reg.item()); n += 1
        history.append((ep, loss_sum/n, reg_sum/n))
        if ep == 1 or ep % log_every == 0 or ep == num_epochs:
            print(f'ep {ep:3d}/{num_epochs}  loss {loss_sum/n:.4e}  reg {reg_sum/n:.4e}  reg/loss {reg_sum/max(loss_sum,1e-12):.2f}')
    return model, history

In [12]:
# run with the config's value first
print('== training with hdr_ff_factor = 0.4 (config default) ==')
model_04, hist_04 = run_training(hdr_ff_factor=0.4, num_epochs=10)

== training with hdr_ff_factor = 0.4 (config default) ==
ep   1/10  loss 2.6237e+00  reg 4.4477e-03  reg/loss 0.00
ep   5/10  loss 4.3762e-01  reg 7.6805e-04  reg/loss 0.00
ep  10/10  loss 2.6070e-01  reg 1.1860e-04  reg/loss 0.00


In [13]:
# now with factor=0 which is what NIK_MRI actually runs
print('== training with hdr_ff_factor = 0.0 (matches NIK_MRI) ==')
model_00, hist_00 = run_training(hdr_ff_factor=0.0, num_epochs=10)

== training with hdr_ff_factor = 0.0 (matches NIK_MRI) ==
ep   1/10  loss 2.5636e+00  reg 0.0000e+00  reg/loss 0.00
ep   5/10  loss 2.9804e-01  reg 0.0000e+00  reg/loss 0.00
ep  10/10  loss 1.7701e-01  reg 0.0000e+00  reg/loss 0.00


## 8. inference and per-coil + combined plots
plotting per-coil BEFORE combine. if only the combined img has the blob then its a csm orientation problem, if every coil has it then k-space is bad.

In [14]:
@torch.no_grad()
def recon(model, t_frame=0):
    kpred = predict_cartesian_kspace(
        model, nt=T, nc=C, nx=img_size[0], ny=img_size[1],
        device=device, chunk_size=131072,
        time_coords=data_meta['time_coords'], coil_coords=data_meta['coil_coords'],
    )  # (T, C, nx, ny) complex
    coil_imgs = ifft2c(kpred)  # (T, C, nx, ny) complex
    combined = coil_combine(coil_imgs, coil_maps)  # (T, nx, ny) complex
    return kpred[t_frame].cpu(), coil_imgs[t_frame].cpu(), combined[t_frame].cpu()

def plot_recon(title, k_t0, coil_t0, combined_t0, coils_to_show=6):
    n = min(coils_to_show, coil_t0.shape[0])
    fig, axes = plt.subplots(2, n + 1, figsize=(2.1*(n+1), 4.5))
    axes[0, 0].imshow(np.log(np.abs(k_t0[0].numpy()) + 1e-4), cmap='viridis')
    axes[0, 0].set_title(f'{title}\nlog|k| coil0'); axes[0, 0].axis('off')
    axes[1, 0].imshow(np.abs(combined_t0.numpy()), cmap='gray')
    axes[1, 0].set_title('|combined|'); axes[1, 0].axis('off')
    for c in range(n):
        axes[0, c+1].imshow(np.abs(coil_t0[c].numpy()), cmap='gray')
        axes[0, c+1].set_title(f'|coil {c}|'); axes[0, c+1].axis('off')
        axes[1, c+1].imshow(np.angle(coil_t0[c].numpy()), cmap='twilight')
        axes[1, c+1].set_title(f'∠coil {c}'); axes[1, c+1].axis('off')
    plt.tight_layout(); plt.show()

k04, ci04, comb04 = recon(model_04)
plot_recon('factor=0.4', k04, ci04, comb04)

k00, ci00, comb00 = recon(model_00)
plot_recon('factor=0.0', k00, ci00, comb00)

/home/rnga/vvpshenov/tmp/ipykernel_983397/3051575707.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 9. side-by-side with reference

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(np.abs(sense_ref), cmap='gray'); axes[0].set_title('ref (nufft + sense)'); axes[0].axis('off')
axes[1].imshow(np.abs(comb04.numpy()), cmap='gray'); axes[1].set_title('nn, factor=0.4'); axes[1].axis('off')
axes[2].imshow(np.abs(comb00.numpy()), cmap='gray'); axes[2].set_title('nn, factor=0.0'); axes[2].axis('off')
plt.tight_layout(); plt.show()

# print central peak intensity ratio as a proxy for 'is the middle blob present'
def center_ratio(img):
    a = np.abs(img); h, w = a.shape; cy, cx = h//2, w//2
    center = a[cy-5:cy+5, cx-5:cx+5].mean()
    rim = np.concatenate([a[:10].ravel(), a[-10:].ravel(), a[:,:10].ravel(), a[:,-10:].ravel()]).mean()
    return center / (rim + 1e-12)

print(f'ref       center/rim ratio = {center_ratio(sense_ref):.2f}')
print(f'factor=04 center/rim ratio = {center_ratio(comb04.numpy()):.2f}')
print(f'factor=00 center/rim ratio = {center_ratio(comb00.numpy()):.2f}')
print('-> if factor=0 drops this closer to the ref value, the regulariser was the problem')

ref       center/rim ratio = 4.45
factor=04 center/rim ratio = 118.66
factor=00 center/rim ratio = 110.70
-> if factor=0 drops this closer to the ref value, the regulariser was the problem


/home/rnga/vvpshenov/tmp/ipykernel_983397/2044321705.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 10. quick residual check at seen coords
if the model cant even fit training points, smth is broken beyond the loss weighting.

In [16]:
@torch.no_grad()
def residuals(model, n=20000):
    idx = torch.randperm(coords.shape[0])[:n]
    x = coords[idx].to(device)
    y = targets[idx].to(device)
    pred = model(x).squeeze(-1)
    y_c = y.squeeze(-1).to(pred.dtype)
    err = (pred - y_c).abs().cpu().numpy()
    y_abs = y_c.abs().cpu().numpy()
    r = (x[:, 2]**2 + x[:, 3]**2).sqrt().cpu().numpy()
    return r, y_abs, err

r04, yA04, e04 = residuals(model_04)
r00, yA00, e00 = residuals(model_00)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].scatter(r04, e04, s=0.5, alpha=0.3, label='f=0.4')
ax[0].scatter(r00, e00, s=0.5, alpha=0.3, label='f=0.0')
ax[0].set_xlabel('|k|'); ax[0].set_ylabel('|err|'); ax[0].set_yscale('log'); ax[0].legend()
ax[0].set_title('abs residual vs |k|')
ax[1].scatter(yA04, e04, s=0.5, alpha=0.3, label='f=0.4')
ax[1].scatter(yA00, e00, s=0.5, alpha=0.3, label='f=0.0')
ax[1].set_xlabel('|target|'); ax[1].set_ylabel('|err|'); ax[1].set_xscale('log'); ax[1].set_yscale('log')
ax[1].legend(); ax[1].set_title('abs residual vs |target|')
plt.tight_layout(); plt.show()

/home/rnga/vvpshenov/tmp/ipykernel_983397/982410289.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## notes
- if section 2 shows kx,ky wandering within each n_slices group -> stack-of-stars assumption is wrong, fix ifft1d_kz_to_z ordering
- if section 3 shows kx/ky not centered at 0 or not spanning ~[-1,1] -> trajectory scaling is off
- if section 6 reference already blobby -> issue is pipeline (ifft, csm orientation, traj), not network
- if section 8 per-coil images are fine but combined has the blob -> csm orientation/flip problem
- if factor=0.0 looks much better in section 9 -> the hdr_ff regulariser was suppressing high-freq content and low-passing the recon

## 12. data-format checks + output-collapse test
checks 1, 3, 6 from the debug guide. does the data layout actually match what nik expects? is the net collapsed?

In [17]:
# === data-format checks from the guide (1, 3, 6) + output-collapse test ===

# CHECK 6 from guide: where is the highest-|y| sample? should be near (kx=0, ky=0)
tgt_abs = targets.abs().squeeze(-1)
top = torch.topk(tgt_abs, 5)
print('top 5 highest-|y| training points (kx,ky should be near 0):')
print(f"{'rank':>4} {'|y|':>12} {'t':>7} {'coil':>7} {'kx':>10} {'ky':>10}")
for r, i in enumerate(top.indices):
    print(f'{r:>4d} {tgt_abs[i].item():12.4g} '
          f'{coords[i,0].item():+7.3f} {coords[i,1].item():+7.3f} '
          f'{coords[i,2].item():+10.4f} {coords[i,3].item():+10.4f}')

# CHECK 3 extended: per-(t,c) max — global max-abs normalization squishes weak (t,c) slices.
# in DCE the bolus passage makes some frames MUCH brighter than others.
pts_tc = data_meta['points_per_time_coil']
dc_mag = torch.zeros(T, C)
for ti in range(T):
    for ci in range(C):
        s = (ti*C + ci) * pts_tc
        dc_mag[ti, ci] = targets[s:s+pts_tc].abs().max()
print('\nper-(t,c) max |target| after global max-abs normalization:')
print(dc_mag.numpy())
print(f'dynamic range across (t,c): {dc_mag.max().item()/(dc_mag.min().item()+1e-12):.2f}x')
print('  >3x  means many slices train at sub-unity magnitudes and HDR relative-error reduces their gradient.')

# OUTPUT-COLLAPSE TEST (from the guide)
@torch.no_grad()
def collapse_test(model, label):
    idx = torch.randperm(coords.shape[0])[:50000]
    x = coords[idx].to(device)
    pred = model(x).squeeze(-1)
    pr, pi = pred.real, pred.imag
    ratio_r = pr.std().item() / (abs(pr.mean().item()) + 1e-12)
    ratio_i = pi.std().item() / (abs(pi.mean().item()) + 1e-12)
    print(f'{label}:  real mean={pr.mean().item():+.3e} std={pr.std().item():.3e}  std/|mean|={ratio_r:.3f}')
    print(f'{label}:  imag mean={pi.mean().item():+.3e} std={pi.std().item():.3e}  std/|mean|={ratio_i:.3f}')
    print(f'{label}:  |pred| mean={pred.abs().mean().item():.3e}  max={pred.abs().max().item():.3e}')

print('\n--- output-collapse test ---')
collapse_test(model_04, 'factor=0.4')
collapse_test(model_00, 'factor=0.0')
print('-> if std/|mean| < 0.01 network has collapsed to ~constant (IFFT(const)=delta at center).')

top 5 highest-|y| training points (kx,ky should be near 0):
rank          |y|       t    coil         kx         ky
   0            1  +0.400  +0.143    -0.0003    +0.0032
   1            1  +0.400  +0.143    +0.0003    -0.0032
   2       0.9987  +0.400  +0.143    -0.0003    -0.0032
   3       0.9987  +0.400  +0.143    +0.0003    +0.0032
   4       0.9984  +0.400  +0.143    -0.0001    -0.0032

per-(t,c) max |target| after global max-abs normalization:
[[0.4867094  0.49712086 0.4953641  0.5463188  0.5616926  0.5458066
  0.49449098 0.49685982]
 [0.5132638  0.52280104 0.51920027 0.5761957  0.59233797 0.5768445
  0.52437866 0.524211  ]
 [0.8250337  0.84322536 0.85372037 0.93074536 0.95213974 0.9261231
  0.8405534  0.83941096]
 [0.8665049  0.8912812  0.91153955 0.979536   1.0000001  0.9681248
  0.87255794 0.8796751 ]
 [0.8107436  0.83522594 0.85064083 0.9146705  0.9356479  0.90414804
  0.8163671  0.8246143 ]]
dynamic range across (t,c): 2.05x
  >3x  means many slices train at sub-unity magn

## 13. what does predicted k-space look like?
a flat/gaussian prediction -> blurry blob in image. a real k-space-like prediction -> anatomy.

In [18]:
# what does the network think k-space looks like?
# if its a narrow gaussian-like bump at DC -> IFFT gives a broad central blob -> 'peak in middle'
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for i, (label, k) in enumerate([('factor=0.4', k04), ('factor=0.0', k00)]):
    km = np.abs(k[0].numpy())
    axes[i, 0].imshow(np.log(km + 1e-4), cmap='viridis')
    axes[i, 0].set_title(f'{label} log|kpred| coil0'); axes[i, 0].axis('off')
    cy, cx = km.shape[0]//2, km.shape[1]//2
    axes[i, 1].plot(km[cy]); axes[i, 1].set_yscale('log'); axes[i, 1].grid(alpha=0.3)
    axes[i, 1].set_title('|kpred| horizontal thru DC')
    axes[i, 2].plot(km[:, cx]); axes[i, 2].set_yscale('log'); axes[i, 2].grid(alpha=0.3)
    axes[i, 2].set_title('|kpred| vertical thru DC')
plt.tight_layout(); plt.show()
print('-> flat profile = const prediction (collapse).  narrow spike at DC = low-freq-only fit.')
print('-> broad shape with structure = proper fit.')

/home/rnga/vvpshenov/tmp/ipykernel_983397/361632585.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


-> flat profile = const prediction (collapse).  narrow spike at DC = low-freq-only fit.
-> broad shape with structure = proper fit.


## 14. isolate single (t, coil, z) fit
can nik fit just ONE frame + ONE coil + this z-slice as a static 2d radial problem? if yes, the blob is from temporal/coil conditioning being too weak. if no, its a fundamental arch/loss/optim issue.

In [19]:
# isolate single (t,c,z) — does nik even fit a static 2d radial image?
# if this gives a delta/blob too, the problem is fundamental (arch/loss/optim).
# if it gives a real image, the multi-(t,c) conditioning is the bottleneck.
from torch.utils.data import DataLoader as _DL

def make_single_tc_dataset(t_idx=0, c_idx=0):
    pts = data_meta['points_per_time_coil']
    start = (t_idx*C + c_idx) * pts
    end = start + pts
    sc = coords[start:end].clone()
    st = targets[start:end].clone()
    s = st.abs().amax().clamp_min(1e-8)
    return sc, st / s, float(s)

sc, st, ss = make_single_tc_dataset(0, 0)
print(f'single (t=0, c=0): N={len(sc)}, local scale={ss:.4g}')
sub_ds = DynamicSliceDataset(sc, st)
sub_loader = _DL(sub_ds, batch_size=30000, shuffle=True, num_workers=0, pin_memory=(device.type=='cuda'))

# smaller + higher lr since single slice is easy
torch.manual_seed(0)
m1 = NIK_MRI_SIREN_REIM(coord_dim=4, feature_dim=256, num_layers=6, out_dim=1, omega_0=30.0, ff_scale=1.0).to(device)
opt = torch.optim.Adam(m1.parameters(), lr=3e-4)
crit = HDRLossFF(sigma=1.0, eps=1e-2, factor=0.0)
for ep in range(1, 41):
    losses = []
    for b in sub_loader:
        x = b['coords'].to(device); y = b['targets'].to(device)
        opt.zero_grad(set_to_none=True)
        loss, _ = crit(m1(x), y, x)
        loss.backward(); opt.step()
        losses.append(loss.item())
    if ep == 1 or ep % 5 == 0:
        print(f'ep {ep:3d}  loss {np.mean(losses):.4e}')

# reconstruct at (t=0, c=0)
@torch.no_grad()
def recon1(m, t_val, c_val):
    nx, ny = img_size
    kxs = torch.linspace(-1, 1-2/nx, nx, device=device)
    kys = torch.linspace(-1, 1-2/ny, ny, device=device)
    kxg, kyg = torch.meshgrid(kxs, kys, indexing='ij')
    x = torch.stack([torch.full_like(kxg, t_val).flatten(),
                     torch.full_like(kxg, c_val).flatten(),
                     kxg.flatten(), kyg.flatten()], dim=1)
    mask = (kxg**2 + kyg**2 < 1).flatten()
    preds = []
    for s_ in range(0, x.shape[0], 131072):
        preds.append(m(x[s_:s_+131072]).squeeze(-1))
    k = torch.cat(preds).view(nx, ny)
    k[~mask.view(nx, ny)] = 0
    return k

t0 = float(data_meta['time_coords'][0])
c0 = float(data_meta['coil_coords'][0])
k1 = recon1(m1, t0, c0).cpu()
img1 = torch.fft.fftshift(torch.fft.ifft2(torch.fft.ifftshift(k1), norm='ortho'))
img_ref1 = nufft2d_recon(k_img_space, traj_t, t_frame=0, coil_idx=0, z_slice_idx=z_slice_idx,
                         scales=scales, img_size=img_size, n_slices=n_slices, return_complex=True)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(np.log(np.abs(k1.numpy()) + 1e-4), cmap='viridis'); axes[0].set_title('single-(t0,c0) log|kpred|')
axes[1].imshow(np.abs(img1.numpy()), cmap='gray'); axes[1].set_title('|img| from single fit')
axes[2].imshow(np.abs(img_ref1), cmap='gray'); axes[2].set_title('|img| ref (nufft same t,c)')
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

single (t=0, c=0): N=66456, local scale=0.4867
ep   1  loss 1.3246e+00
ep   5  loss 1.6871e+00
ep  10  loss 4.2078e+00
ep  15  loss 1.0369e+01
ep  20  loss 3.3302e+00
ep  25  loss 3.9073e+00
ep  30  loss 1.0133e+01
ep  35  loss 9.5294e+00
ep  40  loss 3.1496e+00


/home/rnga/vvpshenov/tmp/ipykernel_983397/50157715.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 15. decision tree for next steps
- if **collapse test** says `std/|mean| < 0.01` -> optimiser failed. try lr=1e-3, AdamW, or check grad norms. not a data issue.
- if **DC-location** shows highest-|y| NOT at (kx,ky)≈(0,0) -> trajectory axis sign/swap bug. fix and retry.
- if **per-(t,c) max** spans >3x -> global max-abs crushes low-signal frames. switch to per-(t,c) renorm or use 0.99-quantile scale.
- if **predicted log|k|** is a narrow gaussian around DC -> net only fit low freqs (undertrained or capacity too low). train longer or widen model.
- if the **single-(t,c) fit** at section 14 gives a clean image -> problem is the t,coil conditioning (need more expressive encoding for these). if it ALSO gives a blob -> core arch/loss issue.

## 16. long single-slice fit with MSE + random batches
strip the problem down. same NIK model, single (t,c), but:
- pure MSE (no HDR denom weirdness)
- `torch.randint` random sampling instead of dataloader
- 10000 gradient steps (paper-ish scale, single slice)
if THIS still gives a blob, its arch/loss/fourier-scale. if it gives an image, the full recipe just needs much longer training.

In [20]:
# === long single-(t,c,z) fit: MSE + random-batch sampling + 10k steps ===
# move single-slice data to gpu once, then sample with torch.randint
sc_gpu = sc.to(device)
st_gpu = st.to(device)  # complex64
N = sc_gpu.shape[0]
print(f'N single-slice points: {N}')

def mse_complex(pred, target):
    # pred: complex (B,1) or (B,); target: complex (B,1) or (B,)
    p = pred.reshape(-1)
    t = target.reshape(-1)
    return ((p - t).abs() ** 2).mean()

def train_single_mse(steps=10000, batch=30000, lr=3e-4, ff_scale=1.0, feature_dim=256, depth=6, log_every=500):
    torch.manual_seed(1)
    m = NIK_MRI_SIREN_REIM(coord_dim=4, feature_dim=feature_dim, num_layers=depth,
                           out_dim=1, omega_0=30.0, ff_scale=ff_scale).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    losses = []
    for s in range(1, steps + 1):
        idx = torch.randint(0, N, (batch,), device=device)
        x = sc_gpu[idx]; y = st_gpu[idx]
        opt.zero_grad(set_to_none=True)
        pred = m(x)
        loss = mse_complex(pred, y)
        loss.backward(); opt.step()
        losses.append(loss.item())
        if s == 1 or s % log_every == 0 or s == steps:
            print(f'step {s:5d}  loss {np.mean(losses[-log_every:]):.4e}')
    return m, losses

m_long, losses_long = train_single_mse(steps=10000, batch=30000, lr=3e-4)

k_long = recon1(m_long, t0, c0).cpu()
img_long = torch.fft.fftshift(torch.fft.ifft2(torch.fft.ifftshift(k_long), norm='ortho'))

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].plot(losses_long); axes[0].set_yscale('log'); axes[0].set_xlabel('step'); axes[0].set_ylabel('MSE'); axes[0].set_title('training loss')
axes[1].imshow(np.log(np.abs(k_long.numpy()) + 1e-4), cmap='viridis'); axes[1].set_title('log|kpred|'); axes[1].axis('off')
axes[2].imshow(np.abs(img_long.numpy()), cmap='gray'); axes[2].set_title('|img| NIK long MSE'); axes[2].axis('off')
axes[3].imshow(np.abs(img_ref1), cmap='gray'); axes[3].set_title('|img| nufft ref'); axes[3].axis('off')
plt.tight_layout(); plt.show()
print(f'\nfinal loss: {np.mean(losses_long[-200:]):.4e}')
print('collapse check:')
with torch.no_grad():
    preds_check = m_long(sc_gpu[:50000])
    print(f'  std/|mean| real: {preds_check.real.std().item()/(abs(preds_check.real.mean().item())+1e-12):.3f}')
    print(f'  std/|mean| imag: {preds_check.imag.std().item()/(abs(preds_check.imag.mean().item())+1e-12):.3f}')

N single-slice points: 66456
step     1  loss 7.6375e-03
step   500  loss 6.2443e-03
step  1000  loss 6.0548e-03
step  1500  loss 6.1021e-03
step  2000  loss 6.0441e-03
step  2500  loss 6.0273e-03
step  3000  loss 6.0295e-03
step  3500  loss 6.0114e-03
step  4000  loss 6.0099e-03
step  4500  loss 6.0071e-03
step  5000  loss 6.0043e-03
step  5500  loss 6.0311e-03
step  6000  loss 6.0158e-03
step  6500  loss 6.0158e-03
step  7000  loss 6.0345e-03
step  7500  loss 6.0176e-03
step  8000  loss 6.0327e-03
step  8500  loss 5.9859e-03
step  9000  loss 5.9924e-03
step  9500  loss 6.0377e-03
step 10000  loss 6.0305e-03

final loss: 6.0483e-03
collapse check:
  std/|mean| real: 0.539
  std/|mean| imag: 7.127


/home/rnga/vvpshenov/tmp/ipykernel_983397/853605122.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 17. ff_scale sweep
the random-fourier-feature bandwidth controls how much high-frequency content the net can represent. NIK_MRI default (ff_scale=1, from randn with no scaling) is on the low end for a 312x312 image. sweep to see if the blob is "net can't reach high freqs". if ff_scale=3 or 5 gives a sharp image, thats the fix for production runs too.

In [21]:
# ff_scale sweep with long single-slice MSE fits
results_ff = {}
for ffs in [1.0, 3.0, 5.0, 10.0]:
    print(f'\n--- ff_scale = {ffs} ---')
    m_ff, losses_ff = train_single_mse(steps=5000, batch=30000, lr=3e-4, ff_scale=ffs, log_every=1000)
    k_ff = recon1(m_ff, t0, c0).cpu()
    img_ff = torch.fft.fftshift(torch.fft.ifft2(torch.fft.ifftshift(k_ff), norm='ortho'))
    results_ff[ffs] = (k_ff, img_ff, np.mean(losses_ff[-200:]))

fig, axes = plt.subplots(2, len(results_ff) + 1, figsize=(3 * (len(results_ff)+1), 6))
for j, (ffs, (k_ff, img_ff, fl)) in enumerate(results_ff.items()):
    axes[0, j].imshow(np.log(np.abs(k_ff.numpy()) + 1e-4), cmap='viridis')
    axes[0, j].set_title(f'ff_scale={ffs}\nlog|k|, loss={fl:.2e}'); axes[0, j].axis('off')
    axes[1, j].imshow(np.abs(img_ff.numpy()), cmap='gray'); axes[1, j].axis('off')
    axes[1, j].set_title('|img|')
axes[0, -1].imshow(np.log(np.abs(k_img_space[0, :, 0, z_slice_idx, :].cpu().numpy()) + 1e-4),
                   cmap='viridis', aspect='auto'); axes[0, -1].set_title('measured hybrid-space'); axes[0, -1].axis('off')
axes[1, -1].imshow(np.abs(img_ref1), cmap='gray'); axes[1, -1].set_title('nufft ref'); axes[1, -1].axis('off')
plt.tight_layout(); plt.show()


--- ff_scale = 1.0 ---
step     1  loss 7.6375e-03
step  1000  loss 6.1495e-03
step  2000  loss 6.0731e-03
step  3000  loss 6.0284e-03
step  4000  loss 6.0107e-03
step  5000  loss 6.0057e-03

--- ff_scale = 3.0 ---
step     1  loss 8.7400e-03
step  1000  loss 6.0445e-03
step  2000  loss 6.0499e-03
step  3000  loss 6.0213e-03
step  4000  loss 6.0045e-03
step  5000  loss 6.0013e-03

--- ff_scale = 5.0 ---
step     1  loss 7.5126e-03
step  1000  loss 5.9344e-03
step  2000  loss 6.0196e-03
step  3000  loss 6.0207e-03
step  4000  loss 6.0050e-03
step  5000  loss 6.0012e-03

--- ff_scale = 10.0 ---
step     1  loss 8.0224e-03
step  1000  loss 5.5927e-03
step  2000  loss 5.6266e-03
step  3000  loss 5.5673e-03
step  4000  loss 5.7280e-03
step  5000  loss 5.6525e-03


/home/rnga/vvpshenov/tmp/ipykernel_983397/1967145142.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
